# Market Technical Indicators

Calculate the established technical-indicator feature table from the observed AAPL dollar bars. Existing features are reused so repeated notebook runs do not repeat the calculation.

## Process the Data

- **Purpose.** Calculate the established technical-indicator feature set from observed AAPL dollar bars.
- **Key settings.** `lookback=14` dollar bars across the indicator set.
- **Data & decision.** Reuse the existing feature artifact to keep repeated runs stable and avoid recomputation.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_preprocessing.market_technical_indicators import (
    save_market_technical_indicators,
)

PROJECT_ROOT = Path.cwd().resolve().parents[1]
feature_dir = PROJECT_ROOT / "data/research_data/market/features"
period = "2025-01-01_2025-12-31"
dollar_bar_path = feature_dir / f"aapl_dollar_bar_{period}.parquet"
technical_path = feature_dir / f"aapl_dollar_bar_technical_{period}.parquet"

if not technical_path.is_file():
    save_market_technical_indicators(
        data_path=dollar_bar_path,
        window=14,
        output_path=technical_path,
    )

technical_features = pd.read_parquet(technical_path)
identifier_columns = ["start", "end", "symbol"]
feature_columns = [
    column for column in technical_features.columns
    if column not in identifier_columns
]
technical_path

PosixPath('/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/market/features/aapl_dollar_bar_technical_2025-01-01_2025-12-31.parquet')

## Take a Quick Look at the Data Structure

- **Purpose.** Inspect the single-symbol schema and the distributions of 51 numeric indicators.
- **Key settings.** No analytical parameters; read-only inspection.
- **Data & decision.** Replace infinities only in the temporary inspection frame and leave the persisted feature table unchanged.

In [2]:
technical_features.head()

,start,end,symbol,McClellan Oscillator,Advancers - Decliners,On-Balance Volume,Accumulation/Distribution Line,Chaikin Oscillator,New Highs - New Lows,Money Flow Index,...,Bollinger Band Middle,Bollinger Band Lower,True Range,Average True Range,Keltner Channel Upper,Keltner Channel Middle,Keltner Channel Lower,Donchian Channel Upper,Donchian Channel Middle,Donchian Channel Lower
0,2025-01-02 13:45:58.902948+00:00,2025-01-02 14:30:02.119118+00:00,AAPL,0.0000,0.000,0.0,-3932.827338,0.0000,0,NaN,...,NaN,NaN,1.390,1.3900,251.4300,248.6500,245.8700,NaN,NaN,NaN
1,2025-01-02 14:30:02.129080+00:00,2025-01-02 14:30:04.394582+00:00,AAPL,12.4305,248.610,-4031.0,-5363.182177,-455.1129,-1,NaN,...,NaN,NaN,0.155,0.7725,250.1897,248.6447,247.0997,NaN,NaN,NaN
2,2025-01-02 14:30:04.394604+00:00,2025-01-02 14:30:09.443442+00:00,AAPL,23.0062,248.805,35.0,-2215.311209,401.6738,1,NaN,...,NaN,NaN,0.310,0.6183,249.9027,248.6660,247.4294,NaN,NaN,NaN
3,2025-01-02 14:30:09.575397+00:00,2025-01-02 14:30:12.024260+00:00,AAPL,31.9271,248.680,-4053.0,-3686.991209,247.3998,0,NaN,...,NaN,NaN,0.250,0.5262,249.7204,248.6679,247.6154,NaN,NaN,NaN
4,2025-01-02 14:30:12.024272+00:00,2025-01-02 14:30:40.602912+00:00,AAPL,39.3871,248.525,-8172.0,-4422.526923,-72.2373,-1,NaN,...,NaN,NaN,0.280,0.4770,249.6029,248.6489,247.6949,NaN,NaN,NaN


In [3]:
technical_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 80213 entries, 0 to 80212
Data columns (total 54 columns):
 #   Column                                    Non-Null Count  Dtype              
---  ------                                    --------------  -----              
 0   start                                     80213 non-null  datetime64[us, UTC]
 1   end                                       80213 non-null  datetime64[us, UTC]
 2   symbol                                    80213 non-null  str                
 3   McClellan Oscillator                      80213 non-null  float64            
 4   Advancers - Decliners                     80213 non-null  float64            
 5   On-Balance Volume                         80213 non-null  float64            
 6   Accumulation/Distribution Line            80213 non-null  float64            
 7   Chaikin Oscillator                        80213 non-null  float64            
 8   New Highs - New Lows                      80213 non-null  int64    

In [4]:
technical_features["symbol"].value_counts(dropna=False)

symbol
AAPL    80213
Name: count, dtype: int64

In [5]:
technical_features[feature_columns].describe()

,McClellan Oscillator,Advancers - Decliners,On-Balance Volume,Accumulation/Distribution Line,Chaikin Oscillator,New Highs - New Lows,Money Flow Index,Williams %R,Aroon Indicator Up,Aroon Indicator Down,...,Bollinger Band Middle,Bollinger Band Lower,True Range,Average True Range,Keltner Channel Upper,Keltner Channel Middle,Keltner Channel Lower,Donchian Channel Upper,Donchian Channel Middle,Donchian Channel Lower
count,80213.000000,80213.000000,8.021300e+04,8.021300e+04,80213.000000,80213.000000,80200.000000,80200.000000,80200.000000,80200.000000,...,80200.000000,80200.000000,80213.000000,80213.00000,80213.000000,80213.000000,80213.000000,80200.000000,80200.000000,80200.000000
mean,0.031297,227.485787,1.630032e+06,2.834583e+06,216.597808,0.005934,50.329655,-49.175529,53.952711,54.788657,...,233.329038,232.783848,0.239829,0.23988,233.811260,233.331501,232.851743,233.857760,233.324953,232.792147
std,4.007916,45.574515,6.919227e+05,1.362886e+06,3359.900830,0.263781,16.040310,32.422500,34.657685,34.580242,...,28.192553,28.298342,0.244057,0.12491,28.114707,28.189724,28.266751,28.101236,28.191331,28.287835
min,-54.962200,0.000000,-6.106000e+04,-3.131070e+04,-22504.077900,-1.000000,0.000000,-100.000000,7.142900,7.142900,...,169.660700,168.441400,0.000000,0.02640,170.198100,169.688800,169.157400,170.410000,169.832500,168.620000
25%,-0.136500,207.660000,1.126200e+06,1.837455e+06,-2000.022600,0.000000,38.798675,-79.330525,21.428600,21.428600,...,209.633200,208.932775,0.130000,0.16070,210.138600,209.620300,209.095800,210.200000,209.610000,208.920000
50%,0.690700,230.780000,1.474584e+06,2.662144e+06,217.211100,0.000000,50.046500,-48.747400,57.142900,57.142900,...,231.719650,231.124000,0.200000,0.21680,232.282300,231.753300,231.188300,232.305000,231.727500,231.090000
75%,1.831800,256.420000,2.158586e+06,4.087485e+06,2460.350800,0.000000,63.376300,-18.852500,85.714300,92.857100,...,257.207725,256.747075,0.300000,0.29070,257.618500,257.210700,256.735900,257.640000,257.200000,256.695000
max,64.023700,288.550000,2.882245e+06,4.978252e+06,31789.828000,1.000000,100.000000,-0.000000,100.000000,100.000000,...,287.929300,287.700000,17.560000,3.83320,288.705100,287.873300,287.487100,288.610000,287.945000,287.660000


In [6]:
technical_features[feature_columns].replace(
    [np.inf, -np.inf], np.nan
).hist(bins=30, figsize=(20, 24))
plt.tight_layout()
plt.show()

/var/folders/1z/bcvql7210c77v6rjkswpzsyr0000gn/T/ipykernel_82309/3467696603.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
